# 🌲 산림 수종 추천 AI — Colab 워크플로

원본 노트북(121셀)을 정리한 깔끔한 실행 버전입니다. `forest_reco` 패키지를 사용합니다.

**준비**: 이 프로젝트 폴더(`wooyoung/`)를 Google Drive에 올리거나 GitHub에서 clone 하세요.
- 임상도/DEM은 `Forest_AI/data/`(51_1.shp, 51_2.shp, gangwon_dem.tif)에 둡니다.
- 데이터가 없으면 아래 **데모 모드**로 합성 데이터를 자동 생성해 그대로 실행됩니다.


## 1. 의존성 설치

In [ ]:
!pip install -q geopandas rasterio pyproj shapely pillow streamlit streamlit-js-eval google-genai piexif
print('설치 완료')

## 2. 프로젝트 경로 연결
아래 `PROJECT_DIR` 를 `forest_reco/` 가 들어있는 폴더로 지정하세요.

In [ ]:
import sys, os
# (선택) Drive 마운트
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# forest_reco 패키지가 있는 폴더 (예: Drive에 업로드한 위치 또는 git clone 경로)
PROJECT_DIR = '/content/drive/MyDrive/Forest_AI/wooyoung'  # ← 수정
if not os.path.isdir(os.path.join(PROJECT_DIR, 'forest_reco')):
    # 대안: GitHub clone
    # !git clone <YOUR_REPO_URL> /content/wooyoung && PROJECT_DIR='/content/wooyoung'
    print('주의: PROJECT_DIR 에 forest_reco 폴더가 없습니다. 경로를 확인하세요.')
sys.path.insert(0, PROJECT_DIR)

# 실데이터 폴더 (없으면 데모 모드 사용)
os.environ['FOREST_RECO_DATA_DIR'] = '/content/drive/MyDrive/Forest_AI/data'
print('PROJECT_DIR =', PROJECT_DIR)

## 3. (선택) 데이터 점검 / DEM 병합

In [ ]:
# 여러 DEM 타일이 있으면 병합
# !python {PROJECT_DIR}/scripts/prepare_data.py merge-dem --in "/content/drive/MyDrive/Forest_AI/data/*.tif" --out /content/drive/MyDrive/Forest_AI/data/gangwon_dem.tif

# 데이터 메타 점검
!python {PROJECT_DIR}/scripts/prepare_data.py inspect --shp $FOREST_RECO_DATA_DIR/51_1.shp --dem $FOREST_RECO_DATA_DIR/gangwon_dem.tif

## 4. 한 지점 분석 (좌표 기반)
실데이터가 있으면 `use_mock=False`, 없으면 `True`(합성 강원권)로 두세요.

In [ ]:
from forest_reco.pipeline import analyze, DataSources

USE_MOCK = True  # 실데이터 있으면 False
src = DataSources(use_mock=USE_MOCK)

res = analyze(lat=37.95, lon=127.66, goal='탄소흡수', audience='산주', sources=src, top_k=6)
print('기후대:', res['site']['climate_zone'], '| 고도:', res['site']['elevation_m'], 'm')
for i, r in enumerate(res['recommendations'], 1):
    print(f"{i}. {r['수종']:<10} {r['적합점수']:>5}  {r['주요근거'][:2]}")
print('
[설명]', res['explanation']['text'])

## 5. 사진(EXIF GPS)으로 분석

In [ ]:
from forest_reco.exif_gps import extract_gps
# 휴대폰 사진 업로드
from google.colab import files
up = files.upload()
path = list(up.keys())[0]
g = extract_gps(path)
print('GPS:', g.lat, g.lon, '| 사유:', g.reason)
if g.ok:
    res = analyze(photo=path, goal='경관조경', sources=src, top_k=5)
    for r in res['recommendations']:
        print(f"{r['수종']:<10} {r['적합점수']}")
else:
    print('사진에 GPS가 없습니다. 좌표를 직접 입력해 분석하세요.')

## 6. Gemini LLM 설명 연결 (선택)
키가 없으면 자동 오프라인 설명이 나옵니다.

In [ ]:
import os
from getpass import getpass
os.environ['GEMINI_API_KEY'] = getpass('Gemini API Key: ')

res = analyze(lat=37.66, lon=127.55, goal='용재생산', audience='연구자', sources=src,
              gemini_api_key=os.environ['GEMINI_API_KEY'], top_k=5)
print('설명 출처:', res['explanation']['source'])
print(res['explanation']['text'])

## 7. 모바일 앱 실행 (Streamlit + cloudflared)

In [ ]:
# cloudflared 다운로드 (최초 1회)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
import subprocess, time, os
os.environ['GEMINI_API_KEY'] = os.environ.get('GEMINI_API_KEY','')
# 앱 실행
subprocess.Popen(['streamlit','run', os.path.join(PROJECT_DIR,'app','streamlit_app.py'),
                  '--server.port','8501','--server.address','0.0.0.0'])
time.sleep(8)
print('앱 시작됨. 아래 셀에서 터널 URL을 여세요.')

In [ ]:
!./cloudflared tunnel --url http://localhost:8501